# 본문 수집 CSV 통계 (Colab용)

`본문_bs4_*.csv` 파일을 기준으로 월별 수집량, 원래 기사 주소 대비 수집률, 결측치, 중복, 카테고리 분포 확인. 한글 파일명이 환경에 따라 다르게 저장될 수 있어 이름을 한 번 정리해서 처리함.

- 입력: `data/본문_bs4_*.csv`, `data/링크_*.json`, `data/본문_bs4_재실패_*.json`
- 출력: 월별 본문 수집량, 쿼리별 요약, 수집률, 결측치, 카테고리 분포
- 저장 파일: `통계_본문수집_월별.csv`, `통계_본문수집_pivot.csv`, `통계_본문수집_query요약.csv`, `통계_본문수집_카테고리.csv`, `통계_본문수집_결측치.csv`


In [21]:
# Colab에서 실행할 때만 사용
from google.colab import drive

drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [22]:
from pathlib import Path
import json
import os
import re
import unicodedata

import pandas as pd

# Colab 기본 경로. 로컬/WSL에서 실행하면 아래 except 경로를 사용
try:
    PROJECT_DIR = Path('/content/drive/MyDrive/Text-data-Analysis_26-Spring')
    if not PROJECT_DIR.exists():
        raise FileNotFoundError
except Exception:
    PROJECT_DIR = Path('/home/carol/Text-data-Analysis_26-Spring')

os.chdir(PROJECT_DIR)  # 상대경로가 프로젝트 기준으로 잡히도록 작업 폴더 변경
print(f'현재 작업 폴더: {Path.cwd()}')

DATA_DIR = PROJECT_DIR / 'data' / 'news'
print(f'DATA_DIR: {DATA_DIR}')

# 분석 대상 필터. True면 *_direct 쿼리(SBS_direct, KBS_direct 등 방송사 직접 수집분)은 통계/시각화에서 제외
EXCLUDE_DIRECT = True

현재 작업 폴더: /content/drive/MyDrive/Text-data-Analysis_26-Spring
DATA_DIR: /content/drive/MyDrive/Text-data-Analysis_26-Spring/data/news


In [23]:
def normalize_text(text):
    return unicodedata.normalize('NFC', str(text))


def normalize_name(path):
    return normalize_text(path.name)


def list_files_normalized(data_dir=DATA_DIR):
    files = []
    for path in data_dir.iterdir():
        if path.is_file():
            files.append((path, normalize_name(path)))
    return files


DATA_FILES = list_files_normalized()
print(f'DATA_DIR 파일 수: {len(DATA_FILES)}')
print('본문 CSV 후보:', sum(name.startswith('본문_bs4_') and name.endswith('.csv') for _, name in DATA_FILES))


def find_file_by_normalized_name(target_name):
    target_name = normalize_text(target_name)
    for path, name in DATA_FILES:
        if name == target_name:
            return path
    return None


def parse_body_filename(path):
    name = normalize_name(path)
    match = re.match(r'^본문_bs4_(.+)_(\d{6})_(\d{6})\.csv$', name)
    if not match:
        return None

    query, start_ymd, end_ymd = match.groups()
    return {
        'query': query,
        'period': f'{start_ymd}_{end_ymd}',
        'start_ym': f'20{start_ymd[:2]}.{start_ymd[2:4]}',
        'start_ymd': start_ymd,
        'end_ymd': end_ymd,
    }


def read_url_count(query, period):
    link_path = find_file_by_normalized_name(f'링크_{query}_{period}.json')
    if link_path is None:
        return None
    with link_path.open('r', encoding='utf-8') as f:
        return len(json.load(f))


def read_fail_count(query, period):
    fail_path = find_file_by_normalized_name(f'본문_bs4_재실패_{query}_{period}.json')
    if fail_path is None:
        return 0
    with fail_path.open('r', encoding='utf-8') as f:
        data = json.load(f)
    return len(data.get('err_idx', []))


DATA_DIR 파일 수: 29
본문 CSV 후보: 6


In [ ]:
summary_rows = []
category_rows = []
missing_rows = []

body_files = sorted(
    [path for path, name in DATA_FILES if name.startswith('본문_bs4_') and name.endswith('.csv') and parse_body_filename(path)],
    key=lambda path: normalize_name(path),
)

if EXCLUDE_DIRECT:
    body_files = [p for p in body_files if not parse_body_filename(p)['query'].endswith('_direct')]
    print(f'EXCLUDE_DIRECT=True: *_direct 쿼리 제외 후 {len(body_files)}개 파일')

if not body_files:
    sample_names = [name for _, name in DATA_FILES[:20]]
    raise ValueError(f'본문_bs4_*.csv 파일을 찾지 못했습니다. DATA_DIR 확인 필요: {DATA_DIR}\n샘플 파일명: {sample_names}')


def normalize_category(cat):
    # '스포츠/축구', '스포츠>야구', '스포츠 일반' 등 스포츠 하위는 모두 '스포츠'로 통합
    if pd.isna(cat):
        return cat
    s = unicodedata.normalize('NFC', str(cat)).strip()
    if s.startswith('스포츠'):
        return '스포츠'
    return s


for csv_path in body_files:
    info = parse_body_filename(csv_path)
    df = pd.read_csv(csv_path, encoding='utf-8-sig')

    url_count = read_url_count(info['query'], info['period'])
    fail_count = read_fail_count(info['query'], info['period'])
    row_count = len(df)
    unique_links = df['link'].nunique() if 'link' in df.columns else None
    duplicate_links = row_count - unique_links if unique_links is not None else None

    summary_rows.append({
        **info,
        'csv_rows': row_count,
        'unique_links': unique_links,
        'duplicate_links': duplicate_links,
        'url_count': url_count,
        'failed_count': fail_count,
        'missing_vs_url': None if url_count is None else url_count - row_count,
        'collection_rate': None if not url_count else row_count / url_count,
        'file_mb': csv_path.stat().st_size / 1024 / 1024,
    })

    for col in ['link', 'pubdate', 'category', 'title', 'body']:
        if col in df.columns:
            missing_rows.append({
                **info,
                'column': col,
                'missing_count': int(df[col].isna().sum() + (df[col].astype(str).str.strip() == '').sum()),
            })

    if 'category' in df.columns:
        cat_series = df['category'].apply(normalize_category)
        counts = cat_series.fillna('').replace('', '(비어있음)').value_counts()
        for category, count in counts.items():
            category_rows.append({**info, 'category': category, 'count': int(count)})

summary_df = pd.DataFrame(summary_rows).sort_values(['query', 'start_ym']).reset_index(drop=True)
category_df = pd.DataFrame(category_rows).sort_values(['query', 'start_ym', 'count'], ascending=[True, True, False]).reset_index(drop=True)
missing_df = pd.DataFrame(missing_rows).sort_values(['query', 'start_ym', 'column']).reset_index(drop=True)

summary_df

In [25]:
# 쿼리 x 월별 본문 수집 건수
body_count_pivot = summary_df.pivot_table(
    index='query',
    columns='start_ym',
    values='csv_rows',
    aggfunc='sum',
    fill_value=0,
)
body_count_pivot['total'] = body_count_pivot.sum(axis=1)
body_count_pivot


start_ym,2026.05,total
query,,
KBS,1514,1514
MBC,1212,1212
SBS,1339,1339


In [26]:
# 원래 기사 주소 개수와 비교해 본문이 얼마나 수집됐는지 확인
rate_pivot = summary_df.pivot_table(
    index='query',
    columns='start_ym',
    values='collection_rate',
    aggfunc='mean',
)
(rate_pivot * 100).round(2)


start_ym,2026.05
query,
KBS,100.0
MBC,100.0
SBS,100.0


In [27]:
# 실패/누락이 있는 파일만 확인
summary_df[
    (summary_df['failed_count'] > 0) |
    (summary_df['missing_vs_url'] > 0) |
    (summary_df['duplicate_links'] > 0)
].sort_values(['failed_count', 'missing_vs_url'], ascending=False)


,query,period,start_ym,start_ymd,end_ymd,csv_rows,unique_links,duplicate_links,url_count,failed_count,missing_vs_url,collection_rate,file_mb


In [28]:
# 쿼리별 요약
query_summary = summary_df.groupby('query', as_index=False).agg(
    csv_rows=('csv_rows', 'sum'),
    unique_links=('unique_links', 'sum'),
    url_count=('url_count', 'sum'),
    failed_count=('failed_count', 'sum'),
    duplicate_links=('duplicate_links', 'sum'),
    file_mb=('file_mb', 'sum'),
)
query_summary['missing_vs_url'] = query_summary['url_count'] - query_summary['csv_rows']
query_summary['collection_rate'] = query_summary['csv_rows'] / query_summary['url_count']
query_summary.sort_values('csv_rows', ascending=False).reset_index(drop=True)


,query,csv_rows,unique_links,url_count,failed_count,duplicate_links,file_mb,missing_vs_url,collection_rate
0,KBS,1514,1514,1514,0,0,2.374118,0,1.0
1,SBS,1339,1339,1339,0,0,3.220684,0,1.0
2,MBC,1212,1212,1212,0,0,2.580447,0,1.0


In [29]:
# 전체 카테고리 분포
category_total = category_df.groupby(['query', 'category'], as_index=False)['count'].sum()
category_total.sort_values(['query', 'count'], ascending=[True, False]).reset_index(drop=True)


,query,category,count
0,KBS,사회,518
1,KBS,정치,366
2,KBS,경제,233
3,KBS,세계,199
4,KBS,생활/문화,98
5,KBS,(비어있음),22
6,KBS,스포츠/general,19
7,KBS,스포츠/kbaseball,14
8,KBS,스포츠/kfootball,14
9,KBS,스포츠/basketball,10


### 카테고리별 시각화

쿼리별 카테고리 분포를 누적 막대그래프(건수)와 100% 비율 막대그래프로 나란히 표시함. 카테고리 순서는 전체 합계 내림차순으로 고정해 두 차트의 색상을 동일하게 맞춤. `EXCLUDE_DIRECT=True`이면 `*_direct` 쿼리는 이미 빠진 상태임.

In [ ]:
import os
import shutil
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm

# ---------- 한글 폰트 설정 ----------
NANUM_PATH = '/usr/share/fonts/truetype/nanum/NanumGothic.ttf'

# Colab/Linux에 NanumGothic 없으면 설치
if not os.path.exists(NANUM_PATH):
    print('NanumGothic 설치 중...')
    os.system('apt-get -y install fonts-nanum > /dev/null 2>&1')
    os.system('fc-cache -fv > /dev/null 2>&1')

# 이전에 저장된 폰트 정보 때문에 한글 폰트가 안 잡히는 경우가 있어 폰트 정보를 다시 불러오도록 정리
_cache_dir = matplotlib.get_cachedir()
if os.path.isdir(_cache_dir):
    for _f in os.listdir(_cache_dir):
        if 'font' in _f.lower():
            try:
                _p = os.path.join(_cache_dir, _f)
                shutil.rmtree(_p) if os.path.isdir(_p) else os.remove(_p)
            except Exception:
                pass

korean_font = None
if os.path.exists(NANUM_PATH):
    fm.fontManager.addfont(NANUM_PATH)
    korean_font = fm.FontProperties(fname=NANUM_PATH).get_name()
else:
    for name in ['Malgun Gothic', 'AppleGothic', 'NanumGothic', 'Noto Sans CJK KR']:
        if name in {f.name for f in fm.fontManager.ttflist}:
            korean_font = name
            break

if korean_font:
    plt.rcParams['font.family'] = korean_font
    plt.rcParams['font.sans-serif'] = [korean_font] + [
        f for f in plt.rcParams['font.sans-serif'] if f != korean_font
    ]
    print(f'한글 폰트: {korean_font}')
else:
    print('경고: 한글 폰트를 찾지 못했습니다. 셀에서 `!apt-get -y install fonts-nanum && rm -rf ~/.cache/matplotlib` 실행 후 런타임 재시작')
plt.rcParams['axes.unicode_minus'] = False

# ---------- 스포츠 하위 카테고리 통합 (이 셀만 따로 실행해도 되도록 준비) ----------
import unicodedata as _ud

def _merge_sports(cat):
    s = _ud.normalize('NFC', str(cat)).strip()
    return '스포츠' if s.startswith('스포츠') else s

cat_data = category_total.copy()
cat_data['category'] = cat_data['category'].apply(_merge_sports)
cat_data = cat_data.groupby(['query', 'category'], as_index=False)['count'].sum()

# ---------- 쿼리별 카테고리 표 만들기 ----------
cat_pivot = cat_data.pivot(index='query', columns='category', values='count').fillna(0)
cat_order = cat_pivot.sum(axis=0).sort_values(ascending=False).index  # 카테고리 순서: 합계 내림차순
cat_pivot = cat_pivot[cat_order]
cat_pivot = cat_pivot.loc[cat_pivot.sum(axis=1).sort_values(ascending=False).index]  # 쿼리: 총량 내림차순

cat_share = cat_pivot.div(cat_pivot.sum(axis=1), axis=0) * 100

fig, axes = plt.subplots(1, 2, figsize=(16, max(4, 0.6 * len(cat_pivot))))

cat_pivot.plot(kind='barh', stacked=True, ax=axes[0], colormap='tab20', width=0.8)
axes[0].set_title('쿼리별 카테고리 분포 (건수)')
axes[0].set_xlabel('수집 건수')
axes[0].set_ylabel('쿼리')
axes[0].invert_yaxis()
axes[0].legend(title='카테고리', bbox_to_anchor=(1.02, 1.0), loc='upper left', fontsize=9)

cat_share.plot(kind='barh', stacked=True, ax=axes[1], colormap='tab20', width=0.8, legend=False)
axes[1].set_title('쿼리별 카테고리 비율 (%)')
axes[1].set_xlabel('비율 (%)')
axes[1].set_ylabel('')
axes[1].invert_yaxis()
axes[1].set_xlim(0, 100)

plt.tight_layout()
plt.show()

In [10]:
# 결측치 확인
missing_df[missing_df['missing_count'] > 0].reset_index(drop=True)


,query,period,start_ym,start_ymd,end_ymd,column,missing_count
0,KBS,260501_260507,2026.05,260501,260507,category,23
1,KBS,260505_260511,2026.05,260505,260511,category,22
2,KBS_direct,260505_260511,2026.05,260505,260511,category,6
3,MBC,260501_260507,2026.05,260501,260507,category,2
4,MBC,260505_260511,2026.05,260505,260511,category,3


In [11]:
# 통계 결과 저장
summary_path = DATA_DIR / '통계_본문수집_월별.csv'
pivot_path = DATA_DIR / '통계_본문수집_pivot.csv'
query_summary_path = DATA_DIR / '통계_본문수집_query요약.csv'
category_path = DATA_DIR / '통계_본문수집_카테고리.csv'
missing_path = DATA_DIR / '통계_본문수집_결측치.csv'

summary_df.to_csv(summary_path, index=False, encoding='utf-8-sig')
body_count_pivot.to_csv(pivot_path, encoding='utf-8-sig')
query_summary.to_csv(query_summary_path, index=False, encoding='utf-8-sig')
category_df.to_csv(category_path, index=False, encoding='utf-8-sig')
missing_df.to_csv(missing_path, index=False, encoding='utf-8-sig')

print(summary_path)
print(pivot_path)
print(query_summary_path)
print(category_path)
print(missing_path)


/content/drive/MyDrive/Text-data-Analysis_26-Spring/data/news/통계_본문수집_월별.csv
/content/drive/MyDrive/Text-data-Analysis_26-Spring/data/news/통계_본문수집_pivot.csv
/content/drive/MyDrive/Text-data-Analysis_26-Spring/data/news/통계_본문수집_query요약.csv
/content/drive/MyDrive/Text-data-Analysis_26-Spring/data/news/통계_본문수집_카테고리.csv
/content/drive/MyDrive/Text-data-Analysis_26-Spring/data/news/통계_본문수집_결측치.csv


## 결과 해석

전체 본문 수집 결과는 원래 기사 주소 69,754개 중 69,693개가 CSV로 저장됨. 전체 수집률은 약 99.91%임. 실패한 61건은 전체의 약 0.09%로 매우 작은 비율이며, 확인 결과 대부분 본문 영역이 없는 속보성 기사라 분석 대상에서 제외해도 전체 기사 수에 비해 적어서 분석 결과에는 큰 영향을 주지 않을 것으로 보임.

쿼리별로 보면 `SK텔레콤`과 `LG유플러스`는 기사 주소 수와 CSV 행 수가 동일해 100% 수집됨. `KT`는 23,907개 중 23,867개, `SKT`는 15,512개 중 15,492개, `LG U+`는 2,781개 중 2,780개가 수집되어 일부 실패가 있었지만 모두 99% 이상 수집률임.

실패가 상대적으로 많았던 구간은 `KT 2025.09` 22건, `SKT 2025.07` 11건, `KT 2025.12` 10건임. 다만 월별 최저 수집률도 `SKT 2025.07`의 약 99.48% 수준이라, 특정 월 자료가 크게 부족해진 정도는 아님.

카테고리는 `IT/과학`이 가장 많고, 그다음 `경제`, `사회`, `정치` 순으로 나타남. 통신사 관련 키워드 수집이라는 점을 고려하면 `IT/과학`과 `경제` 비중이 큰 것은 자연스러운 분포로 볼 수 있음. 카테고리 결측은 566건이며, 본문, 제목, 날짜가 비어 있는 경우는 따로 보이지 않아 본문 분석에는 큰 문제가 없어 보임.

따라서 현재 본문 CSV는 전체 분석에 사용해도 될 정도로 잘 수집된 것으로 보이며, 실패한 기사 주소는 “본문이 없는 속보성 기사 등으로 인해 제외된 사례”로 정리하면 됨.
